# Part 5 — Consensus layer and inter-layer coupling

The per-patient CARNIVAL subnetworks are aggregated into a consensus elementary
layer: a node or edge enters it when it is active in at least `CONSENSUS_MIN_FREQ`
of the cohort. We build one consensus layer per cohort, then add the coupling
edges that tie the same gene across patient layers — the step that makes the
patient subgraphs genuinely Kivelä-multilayer rather than a stack of parallel
graphs. The consensus builder lives in `uc1_common` because `06` reuses it.

In [ ]:
import pickle

import numpy as np
import pandas as pd
from IPython.display import display

import uc1_common as uc

primary_patient_networks = uc.load_carnival()[uc.PRIMARY_LAMBDA]
cohort_definitions = uc.load_cohorts()
G = uc.load()

In [ ]:
consensus_outputs = {}
for cohort_label in uc.SENSITIVITY_COHORT_LABELS:
    members = [p for p in cohort_definitions[cohort_label]
               if p in primary_patient_networks and primary_patient_networks[p]['solver_ok']]
    node_df, edge_df, summary = uc.build_consensus_layer(
        {p: primary_patient_networks[p] for p in members}, uc.CONSENSUS_MIN_FREQ)
    summary['cohort_label'] = cohort_label
    node_df.to_csv(uc.TABLES / f'consensus_nodes_{cohort_label}.csv', index=False)
    edge_df.to_csv(uc.TABLES / f'consensus_edges_{cohort_label}.csv', index=False)
    consensus_outputs[cohort_label] = {
        'node_df': node_df, 'edge_df': edge_df, 'summary': summary,
        'layer_aa': uc.add_consensus_layer_to_graph(G, f'consensus_{cohort_label}', node_df, edge_df),
    }

consensus_summary = pd.DataFrame([x['summary'] for x in consensus_outputs.values()])
consensus_summary.to_csv(uc.TABLES / 'consensus_summary.csv', index=False)
G.history.snapshot('after_consensus_layers')

primary = consensus_outputs[uc.PRIMARY_COHORT_LABEL]
primary_consensus_nodes, primary_consensus_edges = primary['node_df'], primary['edge_df']
primary_consensus_selected = primary_consensus_nodes[primary_consensus_nodes['selected_for_consensus']].copy()

# Invariants: counts are bounded by cohort size, frequencies by 1.0.
assert primary_consensus_nodes['active_count'].max() <= primary['summary']['n_patients']
assert primary_consensus_nodes['patient_frequency'].max() <= 1.0
if not primary_consensus_edges.empty:
    assert primary_consensus_edges['active_count'].max() <= primary['summary']['n_patients']
    assert primary_consensus_edges['patient_frequency'].max() <= 1.0

uc.CONSENSUS_PKL.write_bytes(pickle.dumps(consensus_outputs))
consensus_summary

## Consensus diagnostics

In [ ]:
print(consensus_summary)
print('\nTop consensus nodes (primary cohort):')
display(primary_consensus_nodes.sort_values(
    ['patient_frequency', 'active_count', 'mean_signal'], ascending=False).head(20))
print('\nTop consensus edges (primary cohort):')
display(primary_consensus_edges.sort_values(
    ['patient_frequency', 'active_count', 'mean_signal'], ascending=False).head(20))

## Inter-layer coupling edges

For every consensus gene `v`, add an undirected coupling edge `(v, p_i) ↔ (v, p_j)`
between each pair of patient layers where `v` is active. Re-extracting a single
patient's subgraph *with* coupling then shows more edges than the intra-layer-only
view — the multilayer structure is now real.

In [ ]:
inter_layer_specs = []
for v_row in primary_consensus_selected.itertuples():
    vid = v_row.vertex_id
    activating = [p for p, rec in primary_patient_networks.items()
                  if rec.get('solver_ok') and abs(rec.get('node_signal', {}).get(vid, 0.0)) >= uc.CORNETO_SIG_THRESHOLD]
    for i in range(len(activating)):
        for j in range(i + 1, len(activating)):
            inter_layer_specs.append({'source': (vid, (activating[i],)), 'target': (vid, (activating[j],)),
                                      'edge_directed': False, 'weight': 1.0})

added_eids = G._add_edges_bulk(inter_layer_specs) if inter_layer_specs else []
print(f'Added {len(added_eids):,} inter-layer coupling edges across consensus genes')

sample_patient = sorted(primary_patient_networks)[0]
sg_intra = G.layers.subgraph_from_layer_tuple((sample_patient,))
sg_full = G.layers.subgraph_from_layer_tuple((sample_patient,), include_inter=True)
print(f'Patient {sample_patient}: intra-only |E|={sg_intra.num_edges}  +inter |E|={sg_full.num_edges}')
G.history.snapshot('after_interlayer_coupling')
G.write(str(uc.SNAPSHOT), overwrite=True)